# Hierarchy Recovery — Activation Steering Pilot

Tests whether the counterfactual-necessity null result (n=81, AUC=0.510) was masked by semantic leakage that lexical suppression couldn't catch.

Pipeline, run top to bottom:
1. **Select a fresh pilot sample**
2. **Extract steering directions** -- diff-of-means, per model, at a mid-plateau layer. Fixed an OOM bug here: the original version loaded full 4D activation arrays (all layers x all window tokens) into memory before slicing, which is ~51GB for Qwen-14B alone. Now slices down to the needed layer/token position per-file, before accumulating.
3. **Calibrate alpha** -- REQUIRED before Step 4
4. **Generate steering-suppressed rollouts** -- resumable
5. **Validate**

**Standing commitment**: whatever this shows, the write-up reports all prior attempts alongside this one.

**Requires**: `kill_screen_events.json`, `hidden_reference_labels.csv`, `hidden_reference_labels_round2.csv`, `activations/`, and `activations_filler/`.

## Setup

In [ ]:
!pip install transformers accelerate scikit-learn scipy -q

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

In [ ]:
# ---- scope_flag (inlined; no file to place) ---------------------------------
# Installs a module named `scope_flag` so the gated cells below resolve their
# `import scope_flag` without any file on disk.
#
# SET THIS ONE TOGGLE:
SCOPE_ANALYSIS_ENABLED = False   # False = paper path (default)
#
# Governs scope ANALYSIS AND REPORTING only, never sample construction. Step 1
# below draws its events stratified on the human local/global labels, and no
# flag can change that. See SCOPE_ISOLATION.md section 3.

import sys, types

_sf = types.ModuleType("scope_flag")
_sf.SCOPE_ANALYSIS_ENABLED = SCOPE_ANALYSIS_ENABLED
_sf.SCOPE_BANNER_SHOWN = False

def _scope_banner():
    if _sf.SCOPE_BANNER_SHOWN:
        return _sf.SCOPE_ANALYSIS_ENABLED
    _sf.SCOPE_BANNER_SHOWN = True
    if _sf.SCOPE_ANALYSIS_ENABLED:
        print("\n[scope_flag] SCOPE_ANALYSIS_ENABLED=True — retired scope "
              "sections WILL run. Not reported in the paper.")
    else:
        print("\n[scope_flag] SCOPE_ANALYSIS_ENABLED=False — scope sections "
              "skipped (paper path).")
    return _sf.SCOPE_ANALYSIS_ENABLED

def _skip_note(name):
    print(f"\n[scope_flag] SKIPPED (retired scope analysis): {name}")
    print("[scope_flag]   set SCOPE_ANALYSIS_ENABLED = True above to run it")

_sf.scope_banner = _scope_banner
_sf.skip_note = _skip_note
sys.modules["scope_flag"] = _sf
print(f"scope_flag installed. SCOPE_ANALYSIS_ENABLED = {SCOPE_ANALYSIS_ENABLED}")

## Step 1 — Select a fresh pilot sample

> **On the event draw below.** It is stratified to balance the human local/global annotation categories, so the resulting sample is **not** a random sample of confirmed revision events. This is disclosed rather than switched off: changing it would change which events every downstream result was computed on. It is harmless for the validity controls, which match each revision to a filler at the same marker word in the same model — but the paper must not describe this sample as random. See `SCOPE_ISOLATION.md` §3.


In [ ]:
# =============================================================================
# PILOT EVENT SELECTION -- ACTIVATION STEERING PILOT (fresh sample, new seed)
#
# Same logic as select_pilot_events.py (used for the lexical-suppression pilot
# and full run), but N_PILOT=15 and a NEW seed -- a fresh, independent
# ~15-event sample, not a reproduction of any prior draw. Output goes to
# steering_pilot_events.json, separate from counterfactual_pilot_events.json,
# so neither pipeline's data gets overwritten by the other.
#
# Re-derives real (model, problem_id, char_start, char_end) for a stratified
# subset of the 142 confidently-human-labeled events used in the embedding
# kill screen, matching back against real labels/*.json and traces/*.json.
# Filters to events where the ORIGINAL (with-revision) trace reached the
# correct answer -- necessity is only meaningful there.
# =============================================================================

from pathlib import Path
import json, random
from collections import defaultdict

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
LABELS_DIR = GDRIVE_BASE / "labels"
TRACES_DIR = GDRIVE_BASE / "traces"
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"

KILL_SCREEN_EVENTS_PATH = EXPORT_DIR / "kill_screen_events.json"
HIDDEN_R1_PATH = EXPORT_DIR / "hidden_reference_labels.csv"
HIDDEN_R2_PATH = EXPORT_DIR / "hidden_reference_labels_round2.csv"

N_PILOT = 15  # fresh, small, pre-committed pilot -- same scale as the original lexical-suppression pilot
SEED = 9001  # NEW seed, dedicated to this pilot -- not a reproduction of any prior sample

assert KILL_SCREEN_EVENTS_PATH.exists(), f"{KILL_SCREEN_EVENTS_PATH} not found"

with open(KILL_SCREEN_EVENTS_PATH) as f:
    kill_events = json.load(f)
print(f"Loaded {len(kill_events)} candidate events from the kill screen set.")

# ---- Need model + marker_phrase per (round, eid) to search labels/ with.
# kill_screen_events.json doesn't carry marker_phrase (it was consumed during
# sentence-splitting), so pull it back from the hidden-reference CSVs, which
# do have it, keyed the same way (round, event_id).
import csv

def load_hidden_reference(path, round_num):
    rows = {}
    if not path.exists():
        return rows
    with open(path) as f:
        for r in csv.DictReader(f):
            eid = int(r["event_id"])
            rows[(round_num, eid)] = r
    return rows

hidden = {}
hidden.update(load_hidden_reference(HIDDEN_R1_PATH, 1))
hidden.update(load_hidden_reference(HIDDEN_R2_PATH, 2))
print(f"Hidden reference rows available: {len(hidden)}")

# ---- Index labels/*.json by (model, problem_id) for fast lookup -----------
print("Indexing labels/ ...")
labels_index = {}   # (model, problem_id) -> full label object
for model_dir in LABELS_DIR.iterdir():
    if not model_dir.is_dir():
        continue
    for f in model_dir.glob("*.json"):
        with open(f) as fh:
            obj = json.load(fh)
        pid = obj.get("problem_id", f.stem)
        labels_index[(model_dir.name, pid)] = obj

trace_index = {}
for model_dir in TRACES_DIR.iterdir():
    if not model_dir.is_dir():
        continue
    for f in model_dir.glob("*.json"):
        trace_index[(model_dir.name, f.stem)] = f

# ---- For each kill-screen event, recover (problem_id, char_start, char_end)
# via hidden_reference's problem_id field (it IS carried there), then confirm
# against the label file's revisions list using marker_phrase + verified=true.
candidates = []
n_no_hidden, n_no_label_file, n_no_matching_revision, n_not_verified_correct, n_truncated = 0, 0, 0, 0, 0

for ev in kill_events:
    key = (ev["round"], ev["eid"])
    h = hidden.get(key)
    if h is None:
        n_no_hidden += 1
        continue
    model = ev["model"]
    pid = h["problem_id"]
    marker = h["marker_phrase"]

    label_obj = labels_index.get((model, pid))
    if label_obj is None:
        n_no_label_file += 1
        continue

    # find the specific revision: match marker_phrase, verified, and (as a
    # tie-breaker if multiple candidates share the same marker) prefer the one
    # whose confidence/reason matches hidden reference's recorded values
    revs = [r for r in label_obj.get("revisions", [])
            if r.get("matched_text") == marker and r.get("verified", True)]
    if not revs:
        n_no_matching_revision += 1
        continue
    had_multiple_candidates = len(revs) > 1
    exact_disambiguation_used = False
    if had_multiple_candidates:
        # disambiguate using confidence + reason from hidden reference, if available
        target_conf = h.get("confidence")
        target_reason = h.get("reason")
        exact = [r for r in revs if str(r.get("confidence")) == str(target_conf)
                 and r.get("reason") == target_reason]
        if exact:
            revs = exact
            exact_disambiguation_used = True
        else:
            revs = revs[:1]  # fall back to first candidate -- flagged below as unresolved

    rv = revs[0]
    trace_path = trace_index.get((model, pid))
    if trace_path is None:
        n_no_label_file += 1
        continue
    with open(trace_path) as fh:
        trace_obj = json.load(fh)

    if not trace_obj.get("correct", False):
        n_not_verified_correct += 1
        continue
    if trace_obj.get("truncated", False):
        n_truncated += 1
        continue

    candidates.append({
        "round": ev["round"], "eid": ev["eid"], "model": model, "problem_id": pid,
        "char_start": rv.get("char_start"), "char_end": rv.get("char_end"),
        "marker_phrase": marker, "human_label": ev["label"],
        "disambiguated_from_multiple": had_multiple_candidates,
        "disambiguation_unresolved": had_multiple_candidates and not exact_disambiguation_used,
        "problem": trace_obj.get("problem", ""),
        "answer_gt": trace_obj.get("answer_gt", ""),
        "difficulty": trace_obj.get("difficulty", "hard"),
        "full_response_len": len(trace_obj.get("full_response", "")),
    })

print(f"\nRecovery summary:")
print(f"  no hidden-reference match: {n_no_hidden}")
print(f"  no label file found: {n_no_label_file}")
print(f"  no matching verified revision: {n_no_matching_revision}")
print(f"  original trace not correct (excluded per design decision #3): {n_not_verified_correct}")
print(f"  original trace truncated (excluded, matches original pipeline's own convention): {n_truncated}")
print(f"  eligible candidates: {len(candidates)}")

# ---- Stratified pilot draw: balance local/global as much as available -----
random.seed(SEED)
locals_ = [c for c in candidates if c["human_label"] == "local"]
globals_ = [c for c in candidates if c["human_label"] == "global"]
print(f"\nEligible pool: {len(locals_)} local, {len(globals_)} global")

n_each = N_PILOT // 2
pilot_local = random.sample(locals_, min(n_each, len(locals_)))
pilot_global = random.sample(globals_, min(N_PILOT - len(pilot_local), len(globals_)))
pilot = pilot_local + pilot_global
random.shuffle(pilot)

print(f"Pilot set: {len(pilot)} events ({len(pilot_local)} local, {len(pilot_global)} global)")

# ---- Build prefix_response_text (the response text up to char_start) -----
# NOTE: trace files do NOT store the actual chat-templated prompt string --
# only prompt_tokens (a count). The real prompt must be reconstructed via
# build_prompt(problem, tokenizer, model_name) at generation time, using the
# "problem" and "difficulty" fields captured below -- do not use a stored
# "prompt" field, there isn't a real one to trust.
for c in pilot:
    trace_path = trace_index[(c["model"], c["problem_id"])]
    with open(trace_path) as fh:
        trace_obj = json.load(fh)
    full_response = trace_obj.get("full_response", "")
    cs = c["char_start"]
    c["prefix_response_text"] = full_response[:cs] if cs is not None else None

out_path = EXPORT_DIR / "steering_pilot_events.json"
with open(out_path, "w") as f:
    json.dump(pilot, f, indent=2)
print(f"\nWrote pilot event set: {out_path}")

for c in pilot:
    if c["disambiguation_unresolved"]:
        flag = "  [MULTIPLE CANDIDATES, NO EXACT MATCH -- verify manually before using]"
    elif c["disambiguated_from_multiple"]:
        flag = "  [disambiguated among multiple candidates via confidence+reason match]"
    else:
        flag = ""
    print(f"  {c['model']:<28} {c['problem_id']:<14} label={c['human_label']:<7} "
          f"marker={c['marker_phrase']!r}{flag}")


## Step 2 — Extract steering directions

Diff-of-means from activations already on Drive. Memory-efficient: slices per-file rather than loading full 4D arrays -- needed for Qwen-14B specifically (48 layers x 5120 hidden dim).

In [ ]:
# =============================================================================
# EXTRACT STEERING DIRECTIONS -- diff-of-means from existing saved activations
#
# direction = mean(revision_windows[center]) - mean(filler_windows[center])
#
# Uses revision_windows (activations/) and filler_windows (activations_filler/)
# -- the SAME classes as Task 1's MATCHED contrast (revision-marker vs
# filler-marker, same lexical marker family) -- not the generic
# 'control_windows' field also present in activations/*.npz, which is a
# weaker, non-lexically-matched contrast. Staying consistent with the
# project's established lexical-confound-aware approach.
#
# Layer: does NOT depend on Task 1's saved results JSON (path/existence
# unconfirmed, would be a fragile dependency). Instead uses a simple,
# self-contained heuristic grounded in the already-established finding that
# the accuracy plateau begins ~32-38% of network depth across all four
# models: picks a layer at 65% depth (comfortably inside the plateau, not
# near either edge). n_layers is read directly from the activation array
# shape, so this needs no external file at all.
#
# Output: steering_directions.json -- one entry per model with the direction
# (unit-normalized), the chosen layer, and the raw (pre-normalization) norm
# and mean per-position activation norm at that layer (both useful later for
# picking sensible alpha values relative to the activation scale).
# =============================================================================

from pathlib import Path
import json
import numpy as np

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
ACTIV_BASE = GDRIVE_BASE / "activations"
FILLER_BASE = GDRIVE_BASE / "activations_filler"
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"
OUT_PATH = EXPORT_DIR / "steering_directions.json"

MODELS = ["DeepSeek-R1-Distill-Llama-8B", "DeepSeek-R1-Distill-Qwen-1.5B",
          "DeepSeek-R1-Distill-Qwen-7B", "DeepSeek-R1-Distill-Qwen-14B"]
CENTER = 30                # same center-token convention used throughout this project
PLATEAU_DEPTH_FRACTION = 0.65  # heuristic: comfortably inside the ~32-38%-onset plateau


def peek_n_layers(base_dir, model, key):
    """Reads just ONE file's shape to determine n_layers, without loading
    everything. Needed before we know which layer to slice down to."""
    model_dir = base_dir / model
    if not model_dir.exists():
        return None
    for f in sorted(model_dir.glob("*.npz")):
        with np.load(f) as z:
            if key in z:
                return z[key].shape[1]
    return None


def load_sliced_windows(base_dir, model, key, layer, center):
    """Loads each file's array, immediately slices down to [:, layer, center, :],
    and discards the full array before moving to the next file. Peak memory is
    one file's full array at a time (transient), not every file's full array
    held simultaneously -- the previous version's np.concatenate(arrays) after
    the loop kept everything in memory at once, which is what blew up on
    Qwen-14B (48 layers x 5120 hidden dim -- ~51GB combined across both
    revision and filler arrays, held simultaneously, well past typical Colab
    RAM). This version only ever holds the small (n_events_in_file, hidden_dim)
    slices, which are orders of magnitude smaller."""
    slices = []
    model_dir = base_dir / model
    if not model_dir.exists():
        return None
    for f in sorted(model_dir.glob("*.npz")):
        with np.load(f) as z:
            if key not in z:
                continue
            arr = z[key]                       # full array for this file, transient
            sliced = arr[:, layer, center, :].astype(np.float32).copy()
            slices.append(sliced)
            del arr                            # explicit, though scope-exit would do this anyway
    if not slices:
        return None
    return np.concatenate(slices, axis=0)      # concatenating small slices is cheap


def extract_direction_for_model(model):
    n_layers = peek_n_layers(ACTIV_BASE, model, "revision_windows")
    if n_layers is None:
        print(f"  {model}: no revision_windows files found -- skipping")
        return None

    layer = int(round(PLATEAU_DEPTH_FRACTION * (n_layers - 1)))
    layer = max(0, min(layer, n_layers - 1))

    rev_vecs = load_sliced_windows(ACTIV_BASE, model, "revision_windows", layer, CENTER)
    fil_vecs = load_sliced_windows(FILLER_BASE, model, "filler_windows", layer, CENTER)

    if rev_vecs is None or fil_vecs is None:
        print(f"  {model}: missing revision_windows or filler_windows -- skipping")
        return None

    rev_mean = rev_vecs.mean(axis=0)
    fil_mean = fil_vecs.mean(axis=0)
    raw_direction = rev_mean - fil_mean
    raw_norm = float(np.linalg.norm(raw_direction))

    if raw_norm == 0:
        print(f"  {model}: degenerate zero-norm direction at layer {layer} -- skipping")
        return None

    unit_direction = raw_direction / raw_norm

    # mean activation norm at this layer/position -- reference scale for alpha calibration
    all_vecs = np.concatenate([rev_vecs, fil_vecs], axis=0)
    mean_activation_norm = float(np.linalg.norm(all_vecs, axis=1).mean())

    return {
        "model": model,
        "n_layers": n_layers,
        "layer": layer,
        "layer_fraction": layer / (n_layers - 1),
        "n_revision_events": int(rev_vecs.shape[0]),
        "n_filler_events": int(fil_vecs.shape[0]),
        "raw_direction_norm": raw_norm,
        "mean_activation_norm_at_layer": mean_activation_norm,
        "unit_direction": unit_direction.tolist(),
    }


def main():
    results = {}
    for model in MODELS:
        print(f"\n{model}")
        r = extract_direction_for_model(model)
        if r is not None:
            results[model] = r
            print(f"  layer={r['layer']}/{r['n_layers']} ({r['layer_fraction']:.0%} depth)  "
                  f"n_revision={r['n_revision_events']}  n_filler={r['n_filler_events']}")
            print(f"  raw direction norm={r['raw_direction_norm']:.3f}  "
                  f"mean activation norm at this layer={r['mean_activation_norm_at_layer']:.3f}")
            print(f"  (ratio: direction is {r['raw_direction_norm']/r['mean_activation_norm_at_layer']:.1%} "
                  f"of typical activation magnitude at this layer -- a sanity check, not itself alpha)")

    with open(OUT_PATH, "w") as f:
        json.dump(results, f)
    print(f"\nWrote {len(results)} model directions to {OUT_PATH}")


if __name__ == "__main__":
    main()


## Step 3 — Calibrate steering alpha (required before Step 4)

Read the printed table and fill in `CHOSEN_ALPHA_MULTIPLIER` in Step 4 based on it.

In [ ]:
# =============================================================================
# CALIBRATE STEERING ALPHA
#
# Tests a few candidate steering strengths (including alpha=0, no steering, as
# a baseline) on a small number of events per model. Reports, per (model,
# alpha): answer-extraction success rate (coherence proxy -- if this craters,
# generation broke, not a finding about necessity) and the NATURAL rate of
# backtracking-adjacent language (using the same LEAKAGE_FLAG_WORDS list from
# the lexical-suppression pipeline, observed here, not enforced).
#
# This does NOT auto-select alpha. Per the design doc: read the printed table,
# pick the smallest alpha that meaningfully reduces flagged-language rate
# while keeping the extraction-success rate close to the alpha=0 baseline,
# and record that choice (with this table) before running the real pilot.
#
# Deliberately cheap: short generations (fewer tokens than the real pilot),
# few events, few rollouts. This is exploratory, not a result.
#
# ALPHA SCALE: relative to raw_direction_norm (the natural revision-vs-filler
# signal magnitude), NOT mean_activation_norm_at_layer (ambient scale). These
# differ by ~8-10x for these models -- scaling against the ambient norm meant
# even the gentlest originally-swept value was already several multiples of
# the natural signal. Fixed after review; the sweep is now anchored correctly
# and widened (8 points instead of 4) to actually map the coherence/effect
# tradeoff rather than jumping from "too weak to matter" to "far too strong".
# =============================================================================

from pathlib import Path
import json, re, time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"
STEERING_PILOT_EVENTS_PATH = EXPORT_DIR / "steering_pilot_events.json"
STEERING_DIRECTIONS_PATH = EXPORT_DIR / "steering_directions.json"
CALIBRATION_OUT_PATH = EXPORT_DIR / "steering_alpha_calibration.json"

N_EVENTS_PER_MODEL = 2       # keep this small -- calibration is exploratory
N_ROLLOUTS_PER_CANDIDATE = 2
CALIBRATION_MAX_NEW_TOKENS = 512   # shorter than the real pilot's budget -- speed over completeness
ALPHA_MULTIPLIERS = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
# x raw_direction_norm (the NATURAL magnitude of the revision-vs-filler difference),
# NOT mean_activation_norm_at_layer. raw_direction_norm is only ~10-12% of the ambient
# activation scale for these models -- scaling alpha against the ambient norm meant
# even the smallest previously-swept value (0.5x mean_norm) was already ~4-5x the
# natural signal, not a gentle probe. multiplier=1.0 here means "push by exactly the
# natural class-mean difference" -- a meaningful, interpretable reference point.
# 0.0 = no-steering baseline.

LEAKAGE_FLAG_WORDS = [
    "verify", "double-check", "double check", "seems off", "seems wrong",
    "let me check", "re-examine", "reexamine", "start over", "scratch that",
    "different approach", "different method", "alternative approach", "instead, let's",
    "i realize", "i notice", "hold on", "not quite right", "doesn't seem",
    "let's redo", "recompute", "re-derive", "wait", "actually", "alternatively",
    "hmm", "let me reconsider", "i made a mistake", "let me try again",
]


def extract_answer(text: str) -> str:
    boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        return boxed[-1].strip()
    m = re.search(r'(?:answer is|therefore)[:\s]*([0-9A-Za-z\-\.\/]+)', text, re.IGNORECASE)
    return m.group(1).strip() if m else "UNKNOWN"


def build_prompt(problem: dict, tokenizer, model_name: str) -> str:
    is_reasoning = any(tag in model_name for tag in ["R1", "QwQ"])
    if problem["difficulty"] == "hard":
        instruction = ("Solve the following competition mathematics problem carefully. "
                        "Show your reasoning step by step and verify your answer.")
    else:
        instruction = ("Solve the following mathematics competition problem. "
                        "Show your working step by step.")
    if is_reasoning:
        messages = [{"role": "user", "content": f"{instruction}\n\nProblem: {problem['problem']}"}]
    else:
        messages = [
            {"role": "system", "content": "You are an expert mathematician. Solve problems with "
                                          "clear step-by-step reasoning."},
            {"role": "user", "content": f"{instruction}\n\nProblem: {problem['problem']}"}
        ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def flag_leakage_rate(text):
    text_lower = text.lower()
    return any(w.lower() in text_lower for w in LEAKAGE_FLAG_WORDS)


def get_decoder_layers(model):
    """Locates the decoder layer list for standard HF Llama/Qwen2-style models.
    Both architectures used in this project follow model.model.layers -- fails
    loudly with a clear message if that assumption doesn't hold, rather than
    silently hooking the wrong thing."""
    assert hasattr(model, "model") and hasattr(model.model, "layers"), (
        "Expected model.model.layers (standard HF Llama/Qwen2 layout) -- this model's "
        "architecture doesn't match. Inspect model structure manually before proceeding."
    )
    return model.model.layers


def make_steering_hook(direction_tensor, alpha):
    def hook(module, inputs, output):
        if isinstance(output, tuple):
            hidden_states = output[0] - alpha * direction_tensor
            return (hidden_states,) + output[1:]
        return output - alpha * direction_tensor
    return hook


@torch.no_grad()
def generate_with_steering(model, tokenizer, prefix_text, layer_idx, direction_tensor, alpha,
                            max_new_tokens, seed):
    torch.manual_seed(seed)
    inputs = tokenizer(prefix_text, return_tensors="pt", truncation=False).to(model.device)

    handle = None
    if alpha != 0.0:
        layers = get_decoder_layers(model)
        handle = layers[layer_idx].register_forward_hook(make_steering_hook(direction_tensor, alpha))

    try:
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=0.6, top_p=0.95,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    finally:
        if handle is not None:
            handle.remove()  # always remove, even if generation raises

    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][prompt_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=False)


def main():
    with open(STEERING_PILOT_EVENTS_PATH) as f:
        pilot_events = json.load(f)
    with open(STEERING_DIRECTIONS_PATH) as f:
        directions = json.load(f)

    by_model = {}
    for ev in pilot_events:
        by_model.setdefault(ev["model"], []).append(ev)

    calibration_results = []

    for model_name_short, events in by_model.items():
        if model_name_short not in directions:
            print(f"\n{model_name_short}: no steering direction available -- skipping calibration")
            continue
        dir_info = directions[model_name_short]
        layer_idx = dir_info["layer"]
        natural_norm = dir_info["raw_direction_norm"]  # reference scale for alpha -- see ALPHA_MULTIPLIERS comment
        mean_norm = dir_info["mean_activation_norm_at_layer"]  # reported for context only, not used for alpha

        model_name_full = f"deepseek-ai/{model_name_short}"
        print(f"\n{'='*60}\n{model_name_full}  (layer {layer_idx}, natural signal norm {natural_norm:.2f}, "
              f"ambient activation norm {mean_norm:.2f})\n{'='*60}")

        tokenizer = AutoTokenizer.from_pretrained(model_name_full, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_name_full, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
        model.eval()

        direction_tensor = torch.tensor(dir_info["unit_direction"], dtype=torch.float16, device=model.device)

        calib_events = events[:N_EVENTS_PER_MODEL]
        for mult in ALPHA_MULTIPLIERS:
            alpha = mult * natural_norm
            n_extracted, n_flagged, n_total = 0, 0, 0
            print(f"\n  alpha = {mult} x natural_signal_norm = {alpha:.3f}")
            for ev in calib_events:
                problem_dict = {"problem": ev["problem"], "difficulty": ev["difficulty"]}
                prompt = build_prompt(problem_dict, tokenizer, model_name_full)
                prefix_text = prompt + ev["prefix_response_text"]
                for r in range(N_ROLLOUTS_PER_CANDIDATE):
                    text = generate_with_steering(
                        model, tokenizer, prefix_text, layer_idx, direction_tensor, alpha,
                        CALIBRATION_MAX_NEW_TOKENS, seed=2000 + r)
                    ans = extract_answer(text)
                    n_total += 1
                    if ans != "UNKNOWN":
                        n_extracted += 1
                    if flag_leakage_rate(text):
                        n_flagged += 1
            extraction_rate = n_extracted / n_total
            flagged_rate = n_flagged / n_total
            print(f"    extraction success rate: {extraction_rate:.0%}   "
                  f"backtracking-language rate: {flagged_rate:.0%}   (n={n_total} rollouts)")
            calibration_results.append({
                "model": model_name_short, "alpha_multiplier": mult, "alpha": alpha,
                "extraction_success_rate": extraction_rate, "flagged_language_rate": flagged_rate,
                "n_rollouts": n_total,
            })

        del model
        torch.cuda.empty_cache()

    with open(CALIBRATION_OUT_PATH, "w") as f:
        json.dump(calibration_results, f, indent=2)

    print(f"\n{'#'*70}\n# CALIBRATION SUMMARY -- read this table, then pick alpha manually\n{'#'*70}")
    print("alpha_mult is relative to the NATURAL signal norm (raw_direction_norm), not ambient")
    print("activation magnitude -- 1.0 means 'push by exactly the natural revision-vs-filler")
    print("class-mean difference'.\n")
    print(f"{'Model':<28} {'alpha_mult':>10} {'extraction_rate':>16} {'flagged_rate':>13}")
    for r in calibration_results:
        print(f"{r['model']:<28} {r['alpha_multiplier']:>10} "
              f"{r['extraction_success_rate']:>15.0%} {r['flagged_language_rate']:>12.0%}")
    print(f"\nWrote {CALIBRATION_OUT_PATH}")
    print("\nPick the smallest alpha_multiplier where flagged_rate drops meaningfully below the")
    print("alpha=0 baseline WITHOUT extraction_rate collapsing -- record your choice before Step 4.")


if __name__ == "__main__":
    main()


## Step 4 — Generate steering-suppressed rollouts

**Fill in `CHOSEN_ALPHA_MULTIPLIER` using Step 3's output before running** -- the script refuses to run otherwise. Resumable.

In [ ]:
# =============================================================================
# GENERATE STEERING-SUPPRESSED ROLLOUTS
#
# Same overall shape as generate_counterfactual_rollouts.py (prompt
# reconstruction, answer extraction, resume support) but suppression is via
# the activation-steering hook instead of bad_words_ids. Uses the alpha
# chosen from calibrate_steering_alpha.py's output -- NOT auto-selected here,
# must be filled in below after reviewing the calibration table.
#
# Writes to steering_rollout_results.json -- separate from
# counterfactual_rollout_results.json, so the lexical-suppression results
# stay untouched for comparison.
# =============================================================================

from pathlib import Path
import json, re, time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"
STEERING_PILOT_EVENTS_PATH = EXPORT_DIR / "steering_pilot_events.json"
STEERING_DIRECTIONS_PATH = EXPORT_DIR / "steering_directions.json"
RESULTS_PATH = EXPORT_DIR / "steering_rollout_results.json"

N_ROLLOUTS = 8
ROLLOUT_MAX_NEW_TOKENS = 2048

# ---- REQUIRED: fill in after reviewing calibrate_steering_alpha.py's output table.
# alpha_multiplier is relative to each model's raw_direction_norm (the NATURAL
# revision-vs-filler signal magnitude, saved in steering_directions.json) --
# NOT mean_activation_norm_at_layer (ambient scale, ~8-10x larger). 1.0 means
# "push by exactly the natural class-mean difference".
CHOSEN_ALPHA_MULTIPLIER = {
    # "DeepSeek-R1-Distill-Llama-8B": 1.0,
    # "DeepSeek-R1-Distill-Qwen-1.5B": 1.0,
    # "DeepSeek-R1-Distill-Qwen-7B": 1.0,
    # "DeepSeek-R1-Distill-Qwen-14B": 1.0,
}


def extract_answer(text: str) -> str:
    boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        return boxed[-1].strip()
    m = re.search(r'(?:answer is|therefore)[:\s]*([0-9A-Za-z\-\.\/]+)', text, re.IGNORECASE)
    return m.group(1).strip() if m else "UNKNOWN"


def build_prompt(problem: dict, tokenizer, model_name: str) -> str:
    is_reasoning = any(tag in model_name for tag in ["R1", "QwQ"])
    if problem["difficulty"] == "hard":
        instruction = ("Solve the following competition mathematics problem carefully. "
                        "Show your reasoning step by step and verify your answer.")
    else:
        instruction = ("Solve the following mathematics competition problem. "
                        "Show your working step by step.")
    if is_reasoning:
        messages = [{"role": "user", "content": f"{instruction}\n\nProblem: {problem['problem']}"}]
    else:
        messages = [
            {"role": "system", "content": "You are an expert mathematician. Solve problems with "
                                          "clear step-by-step reasoning."},
            {"role": "user", "content": f"{instruction}\n\nProblem: {problem['problem']}"}
        ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def get_decoder_layers(model):
    assert hasattr(model, "model") and hasattr(model.model, "layers"), (
        "Expected model.model.layers (standard HF Llama/Qwen2 layout) -- this model's "
        "architecture doesn't match. Inspect model structure manually before proceeding."
    )
    return model.model.layers


def make_steering_hook(direction_tensor, alpha):
    def hook(module, inputs, output):
        if isinstance(output, tuple):
            hidden_states = output[0] - alpha * direction_tensor
            return (hidden_states,) + output[1:]
        return output - alpha * direction_tensor
    return hook


@torch.no_grad()
def generate_with_steering(model, tokenizer, prefix_text, layer_idx, direction_tensor, alpha, seed):
    torch.manual_seed(seed)
    inputs = tokenizer(prefix_text, return_tensors="pt", truncation=False).to(model.device)

    handle = None
    if alpha != 0.0:
        layers = get_decoder_layers(model)
        handle = layers[layer_idx].register_forward_hook(make_steering_hook(direction_tensor, alpha))

    try:
        output_ids = model.generate(
            **inputs, max_new_tokens=ROLLOUT_MAX_NEW_TOKENS, do_sample=True,
            temperature=0.6, top_p=0.95,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    finally:
        if handle is not None:
            handle.remove()  # always remove, even if generation raises -- verified in testing

    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][prompt_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=False)


def main():
    assert STEERING_PILOT_EVENTS_PATH.exists(), f"{STEERING_PILOT_EVENTS_PATH} not found -- run select_steering_pilot_events.py first"
    assert STEERING_DIRECTIONS_PATH.exists(), f"{STEERING_DIRECTIONS_PATH} not found -- run extract_steering_directions.py first"

    with open(STEERING_PILOT_EVENTS_PATH) as f:
        pilot_events = json.load(f)
    with open(STEERING_DIRECTIONS_PATH) as f:
        directions = json.load(f)
    print(f"Loaded {len(pilot_events)} pilot events.")

    missing_alpha = [m for m in directions if m not in CHOSEN_ALPHA_MULTIPLIER]
    if missing_alpha:
        raise ValueError(
            f"CHOSEN_ALPHA_MULTIPLIER is missing entries for: {missing_alpha}. "
            f"Review calibrate_steering_alpha.py's output table and fill these in -- "
            f"do not proceed with an unreviewed default."
        )

    by_model = {}
    for ev in pilot_events:
        by_model.setdefault(ev["model"], []).append(ev)

    # ---- resume support, same pattern as the lexical-suppression pipeline ----
    all_results = []
    already_done = set()
    if RESULTS_PATH.exists():
        with open(RESULTS_PATH) as f:
            all_results = json.load(f)
        already_done = {(r["problem_id"], r["model"]) for r in all_results}
        print(f"Resuming from {RESULTS_PATH}: {len(already_done)} events already completed.")

    for model_name_short, events in by_model.items():
        if model_name_short not in directions:
            print(f"\n{model_name_short}: no steering direction available -- skipping entirely")
            continue

        remaining_events = [ev for ev in events if (ev["problem_id"], model_name_short) not in already_done]
        n_skipped = len(events) - len(remaining_events)
        if n_skipped:
            print(f"\n{model_name_short}: {n_skipped}/{len(events)} events already done, skipping those.")
        if not remaining_events:
            print(f"{model_name_short}: all events already done -- skipping model load entirely.")
            continue

        dir_info = directions[model_name_short]
        layer_idx = dir_info["layer"]
        natural_norm = dir_info["raw_direction_norm"]  # matches calibrate_steering_alpha.py's scale
        alpha = CHOSEN_ALPHA_MULTIPLIER[model_name_short] * natural_norm

        model_name_full = f"deepseek-ai/{model_name_short}"
        print(f"\n{'='*60}\n{model_name_full}  ({len(remaining_events)} events remaining)")
        print(f"layer={layer_idx}  alpha_multiplier={CHOSEN_ALPHA_MULTIPLIER[model_name_short]}  alpha={alpha:.3f}\n{'='*60}")

        tokenizer = AutoTokenizer.from_pretrained(model_name_full, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_name_full, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
        model.eval()
        direction_tensor = torch.tensor(dir_info["unit_direction"], dtype=torch.float16, device=model.device)

        for ev in remaining_events:
            print(f"\n  {ev['problem_id']}  (human_label={ev['human_label']})")
            problem_dict = {"problem": ev["problem"], "difficulty": ev["difficulty"]}
            reconstructed_prompt = build_prompt(problem_dict, tokenizer, model_name_full)
            prefix_text = reconstructed_prompt + ev["prefix_response_text"]

            rollout_correct = []
            rollout_texts = []
            t0 = time.time()
            for r in range(N_ROLLOUTS):
                rollout_text = generate_with_steering(
                    model, tokenizer, prefix_text, layer_idx, direction_tensor, alpha, seed=3000 + r)
                model_answer = extract_answer(rollout_text)
                is_correct = model_answer.strip() == ev["answer_gt"].strip()
                rollout_correct.append(is_correct)
                rollout_texts.append(rollout_text)
                print(f"    rollout {r+1}/{N_ROLLOUTS}: ans={model_answer!r} "
                      f"(gt={ev['answer_gt']!r}) correct={is_correct}")
            elapsed = time.time() - t0
            necessity = 1.0 - (sum(rollout_correct) / len(rollout_correct))
            print(f"  necessity={necessity:.3f}  ({elapsed:.1f}s for {N_ROLLOUTS} rollouts)")

            all_results.append({
                "problem_id": ev["problem_id"], "model": model_name_short,
                "human_label": ev["human_label"], "rollout_correct": rollout_correct,
                "rollout_texts": rollout_texts, "alpha": alpha, "layer": layer_idx,
            })
            with open(RESULTS_PATH, "w") as f:
                json.dump(all_results, f, indent=2)

        del model
        torch.cuda.empty_cache()

    print(f"\nWrote {len(all_results)} events' rollout results to {RESULTS_PATH}")


if __name__ == "__main__":
    main()


## Step 5 — Validate necessity against human labels

## ⚠️ Retired — necessity vs human scope labels

This step scores necessity against the human **local/global** labels. That analysis is retired: all three scope operationalizations returned nulls and the paper makes no scope claim.

It is **skipped by default** and left here for the record. The live validity control for this mechanism is the matched-filler comparison, in `Paper2_3_Lexical_Filler_Control.ipynb` (lexical) and `Paper2_4_Steering_Powered_Run.ipynb` (steering).

To run it anyway, set `SCOPE_ANALYSIS_ENABLED = True` in the setup cell.


In [ ]:
import scope_flag
# RETIRED — this step measures necessity against the human local/global SCOPE
# labels. The paper makes no scope claim, so it is skipped by default. The LIVE
# validity control for this mechanism is the matched-filler comparison:
#   lexical  -> Paper2_3_Lexical_Filler_Control.ipynb
#   steering -> Paper2_4_Steering_Powered_Run.ipynb
# Body left byte-identical below (it contains triple-quoted strings, so it is
# guarded here rather than indented).
if not scope_flag.scope_banner():
    scope_flag.skip_note("necessity vs human local/global labels")
    raise SystemExit("Retired scope analysis — skipped by design, not an error.")

# =============================================================================
# COUNTERFACTUAL NECESSITY -- validation step
#
# Consumes rollout results (produced by YOUR suppressed-generation code, which
# depends on your inference setup and isn't included here -- see the design
# doc's "open dependency" section). Expected input format, one JSON list:
#
#   [
#     {"problem_id": "...", "model": "...", "human_label": "local"|"global",
#      "rollout_correct": [true, false, true, ...]},   # one bool per rollout
#     ...
#   ]
#
# Computes necessity = 1 - fraction(rollout_correct == True), then checks
# whether necessity separates human-labeled local vs global -- same AUC /
# Mann-Whitney machinery as the embedding-divergence kill screen, for direct
# comparability between the two approaches.
# =============================================================================

import json
from pathlib import Path
import numpy as np
from sklearn.metrics import roc_auc_score
from scipy.stats import mannwhitneyu

GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
EXPORT_DIR = GDRIVE_BASE / "hand_labeling_export"
ROLLOUT_RESULTS_PATH = EXPORT_DIR / "steering_rollout_results.json"

# Same list as audit_suppression_leaks.py's EXPLICITLY_BANNED -- keep in sync.
EXPLICITLY_BANNED = [
    "wait", "actually", "let me reconsider", "that's wrong", "that is wrong",
    "i made an error", "let me try again", "this doesn't work",
    "this does not work", "i need to reconsider", "hmm", "that's not right",
    "that is not right", "let me rethink", "on second thought",
    "let me go back", "wait, let me", "actually, let me", "let me recheck",
    "let me re-check", "alternatively", "i made a mistake",
    "i think i made a mistake", "let me double check", "let me double-check",
    "hold on",
]


def is_leaked(rollout_text):
    text_lower = rollout_text.lower()
    return any(p.lower() in text_lower for p in EXPLICITLY_BANNED)


def filter_leaked_rollouts(results):
    """Drops individual leaked rollouts (not whole events) before computing
    necessity. Events that lose ALL rollouts to leaks are excluded entirely
    and reported separately -- their necessity would be undefined, not zero."""
    filtered_results = []
    n_rollouts_dropped = 0
    n_events_fully_dropped = 0

    for r in results:
        if "rollout_texts" not in r:
            # no text saved (pre-patch run) -- can't filter, pass through as-is
            filtered_results.append(r)
            continue
        clean_correct = []
        for correct, text in zip(r["rollout_correct"], r["rollout_texts"]):
            if is_leaked(text):
                n_rollouts_dropped += 1
            else:
                clean_correct.append(correct)
        if not clean_correct:
            n_events_fully_dropped += 1
            print(f"  WARNING: {r['problem_id']} lost ALL rollouts to leakage -- excluded entirely")
            continue
        new_r = dict(r)
        new_r["rollout_correct"] = clean_correct
        new_r["n_dropped"] = len(r["rollout_correct"]) - len(clean_correct)
        filtered_results.append(new_r)

    print(f"Filtering: dropped {n_rollouts_dropped} individually-leaked rollouts, "
          f"{n_events_fully_dropped} events fully excluded (lost all rollouts).")
    return filtered_results


def compute_necessity(rollout_correct):
    if len(rollout_correct) == 0:
        return None
    return 1.0 - (sum(rollout_correct) / len(rollout_correct))


def report(results, tier_name):
    print(f"\n{'='*70}\n{tier_name}  (n={len(results)})\n{'='*70}")
    local_nec = [r["necessity"] for r in results if r["human_label"] == "local"]
    global_nec = [r["necessity"] for r in results if r["human_label"] == "global"]
    print(f"  n_local={len(local_nec)}  n_global={len(global_nec)}")
    if len(local_nec) < 2 or len(global_nec) < 2:
        print("  TOO FEW EVENTS IN ONE CLASS -- cannot compute AUC reliably.")
        return None
    print(f"  median necessity -- local: {np.median(local_nec):.3f}   global: {np.median(global_nec):.3f}")
    labels = [1]*len(global_nec) + [0]*len(local_nec)
    scores = global_nec + local_nec
    auc = roc_auc_score(labels, scores)
    u_stat, p_val = mannwhitneyu(global_nec, local_nec, alternative='two-sided')
    print(f"  AUC (necessity predicting global): {auc:.3f}")
    print(f"  Mann-Whitney U p-value: {p_val:.4f}")
    return auc


def main():
    with open(ROLLOUT_RESULTS_PATH) as f:
        results = json.load(f)
    print(f"Loaded {len(results)} events with rollout results.")

    results = filter_leaked_rollouts(results)
    print(f"{len(results)} events remain after filtering.\n")

    for r in results:
        r["necessity"] = compute_necessity(r["rollout_correct"])
        r["n_rollouts"] = len(r["rollout_correct"])

    n_rollouts_seen = set(r["n_rollouts"] for r in results)
    if len(n_rollouts_seen) > 1:
        print(f"NOTE: rollout counts vary across events: {sorted(n_rollouts_seen)}. "
              f"Necessity is a fraction so this is comparable across events, but low "
              f"rollout counts (e.g. <5) make individual necessity scores noisy -- read "
              f"any single event's score with that in mind.")

    auc = report(results, "COUNTERFACTUAL NECESSITY PILOT")

    print(f"\n{'#'*70}\n#  DECISION (per pre-committed pilot design)\n{'#'*70}")
    if auc is None:
        print("Could not compute AUC (insufficient class balance). Check pilot event selection.")
    elif auc >= 0.75:
        print(f"AUC={auc:.3f} -- comfortably separates. PROMISING. Scale to the full 200-event "
              f"set with more rollouts per event before concluding anything stronger.")
    else:
        print(f"AUC={auc:.3f} -- near chance / weak. Two live explanations, not one:")
        print("  (a) necessity genuinely isn't a good scope proxy, or")
        print("  (b) token-banning suppression was too leaky to test it properly.")
        print("  Before shelving: manually read a few suppressed rollouts for events with high")
        print("  necessity -- do they actually look like they avoided the revision's effect, or")
        print("  does the model 'notice' the same issue in different words? That distinguishes")
        print("  (a) from (b) and determines whether activation steering is worth trying next.")


if __name__ == "__main__":
    main()
